In [1]:
import os, re, json
import pandas as pd


# analyse the steering results
success schemas:
- counting 1s and -1s
    - counts of 1s of 2pos
    - counts of -1s of 2neg
- comparing the baseline
    - if baseline is 1, after 2neg, how many becomes -1 or 0?
    - if baseline is 0, 
        - after 2neg, how many becomes -1?
        - after 2pos, how many becomes 1?
    - if baseline is -1, after 2pos, how many becomes 0 or 1?
- counts of 1s for bridge

additionally for quality
- counts of 1s for repetition
- average fluency

In [4]:
# all files and dirs
baseline_path = "/scratch/fmeng/ActAdd/results/gemini_base/"
baseline_llama = "gemini_base_llama_senti+_fl_temp_0.json"
baseline_opt = "gemini_base_opt_senti+_fl_temp_0.json"

result_path = "/scratch/fmeng/ActAdd/results/"
dirs_2pos = [
    "gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_llama_senti+_fl_temp_0_hpt", 
    "gemini_sent_2pos_opt_senti+_fl_temp_0_hpt"]

dirs_2neg = [
    "gemini_2neg_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2neg_llama_senti+_fl_temp_0_hpt",
    "gemini_sent_2neg_opt_senti+_fl_temp_0_hpt"]

dirs_bridge = ["gemini_bridge_llama_bridge+_fl_hpt", "gemini_bridge_opt_bridge+_fl_hpt"]

# baseline
no generated sentence in the baseline is talking about the golden gate bridge

In [3]:
# baseline files have a different structure
def base_stats(base_file):
    """
    return the sentiment labels of the base generation for downstream process
    """
    df = pd.read_json(baseline_path + base_file)
    print("counts of", df["continuation_label"].value_counts())
    print("number of repetitive sentences:", df["repetition"].sum().item())
    print("average perplexity of continuations:", df["fluency"].mean().item())
    return df["continuation_label"]

base_llama_sentimap = base_stats(baseline_llama)
base_opt_sentimap = base_stats(baseline_opt)
base_opt_sentimap

counts of continuation_label
 0    13
 1     5
-1     2
Name: count, dtype: int64
number of repetitive sentences: 5
average perplexity of continuations: 2.557923251390457
counts of continuation_label
 0    9
 1    7
-1    4
Name: count, dtype: int64
number of repetitive sentences: 12
average perplexity of continuations: 3.263213074207306


0     0
1     0
2    -1
3     1
4    -1
5     1
6     0
7     0
8     1
9     1
10    0
11    0
12   -1
13   -1
14    0
15    1
16    1
17    1
18    0
19    0
Name: continuation_label, dtype: int64

# sentiment
temperature 1
## schema 1
counting the number of positive/negative

In [94]:
# process files in the whole directory, counting 
def senti_stats(direction, dir):
    """
    direction: "2pos", "2neg" 
    """
    print("showing result for directory", dir)
    grid_one = pd.DataFrame(0, index=range(32),columns=range(20))
    grid_zero = pd.DataFrame(0, index=range(32),columns=range(20))
    grid_neg = pd.DataFrame(0, index=range(32),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(32),columns=range(20))
    grid_fl = pd.DataFrame(index=range(32),columns=range(20))
    for file_name in os.listdir(f"{result_path}{dir}/"):
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["continuation_label"].value_counts():
                grid_one.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[1]
            if 0 in list_dict["continuation_label"].value_counts():
                grid_zero.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[0]
            if -1 in list_dict["continuation_label"].value_counts():
                grid_neg.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[-1]
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object
    if direction == "2pos":
        print("count of positive continuation ↑")
        display(grid_one.style.background_gradient(cmap='Blues'))
    else:
        print("count of negative continuation ↑")
        display(grid_neg.style.background_gradient(cmap='Blues'))
    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds'))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )
    # gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=0)
    # display(grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped))
    # gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    # display(grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped))



In [95]:
for dir in dirs_2pos:
    senti_stats("2pos", dir)
for dir in dirs_2neg:
    senti_stats("2neg", dir)

showing result for directory gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,7,8,6,6,7,8,5,11,7,3,7,2,3,5,2,2,3,2,5,4
1,6,7,5,4,5,2,4,3,3,4,4,3,2,3,2,2,2,2,2,2
2,8,4,3,5,7,6,8,8,10,9,7,10,8,8,8,8,7,6,7,9
3,5,5,5,6,6,4,6,6,7,12,14,9,13,11,8,8,11,8,10,9
4,7,4,5,10,7,6,5,4,5,6,5,7,8,9,7,8,9,10,13,11
5,6,8,5,6,7,7,6,5,6,6,7,6,7,8,9,8,10,10,11,11
6,7,5,8,6,9,9,8,5,7,7,7,9,11,11,10,10,9,9,9,9
7,4,6,8,8,8,6,7,7,7,5,8,10,6,7,5,4,5,8,7,5
8,7,8,8,8,8,8,10,8,7,9,8,8,9,6,7,5,5,6,5,6
9,6,7,8,6,9,6,10,9,6,8,6,7,6,6,5,7,6,6,5,4


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,5,4,5,7,9,11,9,12,8,16,15,15,17,18,17,18,16,18,16
1,7,7,14,18,18,17,15,11,13,16,16,17,17,17,19,19,18,18,18,18
2,5,7,9,14,12,13,14,10,11,5,10,7,9,11,11,12,15,16,14,16
3,6,6,6,8,8,6,9,10,11,13,12,14,11,16,16,15,16,15,17,16
4,6,7,10,9,12,11,15,18,16,17,19,20,20,19,20,20,20,19,19,20
5,4,4,6,7,10,8,11,11,12,13,16,15,13,14,15,17,16,17,18,19
6,4,9,8,8,12,12,9,11,10,14,11,11,10,13,13,12,16,14,13,15
7,5,3,8,8,11,12,9,10,11,12,13,15,13,14,15,13,15,14,14,13
8,5,5,8,8,10,9,11,12,13,12,12,14,14,13,14,15,16,15,16,14
9,6,5,7,6,9,9,12,10,10,11,11,15,14,15,13,12,11,14,13,17


average perplexity of continuations ↓


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,3,5,5,5,4,6,6,6,6,5,4,4,4,4,2,2,3,3,3
1,4,4,5,5,6,5,7,5,4,1,1,3,3,4,3,4,2,2,2,1
2,4,5,4,5,5,3,3,0,1,1,2,2,1,2,0,0,1,0,0,0
3,4,3,3,5,3,3,1,3,2,0,1,0,0,0,0,0,0,0,0,0
4,4,2,4,2,2,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0
5,4,3,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,4,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,3,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,5,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,4,4,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,10,8,10,10,12,9,10,12,12,12,12,13,11,11,11,10,10,10,10,12
1,10,10,14,14,14,14,16,15,13,13,12,13,12,11,12,8,10,12,12,12
2,10,10,11,7,15,14,16,17,16,18,16,17,18,17,18,17,20,20,20,20
3,9,7,10,12,12,17,15,18,19,14,13,8,8,4,4,3,3,3,3,3
4,10,12,13,15,17,18,20,20,19,20,20,19,19,19,18,19,17,19,17,16
5,12,7,10,16,14,18,14,15,14,15,15,12,14,12,13,11,13,13,10,10
6,9,11,14,18,18,20,19,20,18,14,16,17,17,17,18,17,19,19,18,19
7,11,9,12,17,20,17,19,20,20,19,19,20,19,20,20,19,19,19,19,18
8,11,13,15,18,19,19,20,18,19,19,19,19,17,16,17,15,15,15,12,14
9,12,11,12,19,18,20,20,19,20,20,20,20,19,20,20,19,19,20,20,20


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3.21,3.40,3.68,3.49,3.51,3.77,3.54,3.58,3.53,3.17,3.14,3.03,3.17,3.01,3.05,3.07,3.07,3.17,3.06,3.10
1,3.20,3.53,3.63,3.58,3.38,3.60,3.27,3.29,3.32,127.01,126.98,27.43,27.41,27.50,27.73,151.57,108.42,108.51,108.17,108.41
2,3.32,3.49,3.41,3.83,127.61,3.22,2.99,2.71,2.93,2.56,2.69,2.85,3.53,3.56,3.30,3.31,3.36,3.02,2.76,16.25
3,3.45,3.28,3.26,3.53,3.23,3.04,3.40,3.15,2.89,129.97,"9,806.65","82,611.40","3,978,861.52","3,361,772.97","6,194,701.33","151,697,431.25","1,541,399.67","43,858.55","202,289.89","208,520.67"
4,3.34,3.36,3.40,3.09,3.90,3.91,4.13,5.53,3.59,4.27,7.25,"32,761.81","43,398.04",348.01,"79,080,887.58","29,607,927.85","29,691,753.07","20,104.72","16,247,690.48","4,205,337.09"
5,3.36,3.35,3.00,3.67,"115,120.54","8,769.41","1,757,269.48","378,543.96","5,746,442.03","16,808,984.12","82,698,028.67","91,438,657.32","89,110,098.05","95,842,101.85","95,589,450.93","81,356,384.74","168,699,317.74","192,957,370.47","194,915,944.03","198,125,493.61"
6,3.36,3.13,2.96,2.47,2.78,2.49,5.05,"52,202.73","52,853.72","121,811.43","178,391.90","291,104.28","200,715.21","290,544.23","382,637.61","137,208.43","37,690.09","1,060.52","9,467.57","9,716.95"
7,3.19,3.35,3.41,3.17,11.59,5.43,5.31,66.77,255.14,"84,363.33","334,915.85","352,849.89","513,870.84","555,246.37","607,031.09","606,861.88","606,862.36","597,055.46","554,668.00","549,652.38"
8,3.33,3.20,3.12,3.73,18.36,64.50,"2,656,356,520.71","2,656,356,498.40","716,918,245.81","161,070,230.15","716,917,826.02","161,069,940.56","161,069,774.91","161,071,589.42","91,876,622,405.88","91,906,547,127.87","95,176,227,305.96","95,176,227,360.49","95,015,157,711.41","95,015,157,838.44"
9,3.23,3.29,3.68,5.36,4.03,2.72,2.93,2.82,2.76,4.60,5.07,8.73,13.32,12.81,13.72,14.10,15.45,10.86,16.14,13.77


## schema 2
comparing with the baseline results

In [29]:
# base_llama_sentimap[10].item()  # 0, 1, -1
def comparative_stats(dir):
    """
    base_map: base_llama_sentimap or base_opt_sentimap 
    gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt
    """
    print("showing result for directory", dir)    
    grid_success = pd.DataFrame(0, index=range(32),columns=range(20))
    if "2pos" in dir:  # count the tags that are larger than the corresponding one in the base_map
        if "llama" in dir:  # choose which map to use
            for file_name in os.listdir(f"{result_path}{dir}/"):
                layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
                with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                    r_dict = json.load(f)
                for coeff in r_dict:
                    list_dict = pd.DataFrame(r_dict[coeff])
                    coeff = int(coeff)
                    # pointwise compare with base_llama_sentimap
                    successs = list_dict["continuation_label"] > base_llama_sentimap
                    grid_success.loc[layer, coeff-1] = successs.sum()
        if "opt" in dir: 
            for file_name in os.listdir(f"{result_path}{dir}/"):
                layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
                with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                    r_dict = json.load(f)
                for coeff in r_dict:
                    list_dict = pd.DataFrame(r_dict[coeff])
                    coeff = int(coeff)
                    # pointwise compare with base_llama_sentimap
                    successs = list_dict["continuation_label"] > base_opt_sentimap
                    grid_success.loc[layer, coeff-1] = successs.sum()
    elif "2neg" in dir:  # count the tags that are smaller than the corresponding one in the base_map
        if "llama" in dir: 
            for file_name in os.listdir(f"{result_path}{dir}/"):
                layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
                with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                    r_dict = json.load(f)
                for coeff in r_dict:
                    list_dict = pd.DataFrame(r_dict[coeff])
                    coeff = int(coeff)
                    successs = list_dict["continuation_label"] < base_llama_sentimap
                    grid_success.loc[layer, coeff-1] = successs.sum()
        if "opt" in dir: 
            for file_name in os.listdir(f"{result_path}{dir}/"):
                layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
                with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                    r_dict = json.load(f)
                for coeff in r_dict:
                    list_dict = pd.DataFrame(r_dict[coeff])
                    coeff = int(coeff)
                    successs = list_dict["continuation_label"] < base_opt_sentimap
                    grid_success.loc[layer, coeff-1] = successs.sum()
    display(grid_success.style.background_gradient(cmap='Blues'))


In [30]:
for dir in dirs_2pos:
    comparative_stats(dir)
for dir in dirs_2neg:
    comparative_stats(dir)

showing result for directory gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,7,7,5,5,7,7,4,8,5,5,7,3,5,7,3,3,4,4,6,4
1,6,6,6,4,4,2,5,4,4,4,4,4,3,4,3,3,3,3,3,3
2,7,4,4,6,8,5,7,6,7,7,5,8,7,8,9,7,8,7,8,8
3,4,6,6,5,6,6,7,7,7,9,11,10,11,10,7,8,10,7,10,8
4,3,5,5,9,7,5,5,5,7,6,6,7,6,8,6,8,8,8,10,7
5,2,6,5,7,8,6,5,5,5,5,6,6,7,7,8,8,9,9,9,9
6,5,4,6,4,7,8,7,7,7,5,5,6,6,6,6,7,7,7,6,6
7,1,4,3,4,6,7,7,5,6,6,7,9,6,6,4,4,5,6,6,4
8,3,4,5,5,4,5,7,7,6,9,8,8,8,5,5,4,5,5,5,6
9,3,4,4,3,7,4,7,7,6,6,4,5,5,5,5,6,6,6,5,5


showing result for directory gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3,4,5,5,5,4,5,3,4,3,3,3,3,4,4,4,4,4,4,6
1,4,2,4,3,7,6,6,6,7,8,6,8,7,6,9,6,5,5,5,6
2,4,4,8,5,5,4,4,4,3,5,4,4,4,4,4,4,4,4,4,4
3,2,8,4,5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
5,3,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
6,3,5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
7,3,5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
8,1,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
9,2,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4


showing result for directory gemini_sent_2pos_llama_senti+_fl_temp_0_hpt


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,4,7,5,4,4,5,2,2,2,2,4,3,5,4,3,2,5,4,3
1,3,3,3,3,5,4,6,8,7,5,6,7,7,7,7,7,7,7,7,7
2,4,3,7,6,7,8,5,7,7,7,7,7,7,7,7,7,7,7,7,7
3,3,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
4,4,5,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
5,5,6,7,7,7,7,7,7,7,7,6,7,5,7,7,7,7,7,7,7
6,3,5,4,3,4,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7
7,5,7,7,6,7,7,7,7,7,7,7,7,7,7,7,6,7,7,7,7
8,4,5,7,7,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
9,6,6,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,6,6


showing result for directory gemini_sent_2neg_llama_senti+_fl_temp_0_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,3,5,7,7,5,4,5,4,4,4,4,4,4,4,5,4,4,4,4
1,2,4,3,8,5,5,7,5,6,5,5,6,6,6,6,6,6,6,7,7
2,2,5,4,5,7,4,4,4,4,6,6,6,6,7,7,6,7,5,5,4
3,0,2,2,2,2,2,4,5,5,4,3,4,5,5,6,6,6,6,6,5
4,0,1,4,4,5,4,4,4,4,5,7,8,7,6,7,5,5,7,7,5
5,1,4,4,4,5,4,4,5,4,4,6,6,6,5,7,6,5,7,6,4
6,1,2,3,3,3,6,5,5,6,5,5,6,6,6,6,8,8,7,7,4
7,2,4,4,4,4,6,7,7,6,6,4,8,9,5,9,7,10,7,11,9
8,1,2,4,8,6,8,6,7,4,6,7,8,8,7,6,9,6,6,6,6
9,1,1,4,5,6,7,5,5,6,7,7,7,7,8,5,7,6,4,3,5


showing result for directory gemini_sent_2neg_opt_senti+_fl_temp_0_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,4,9,7,8,7,6,6,6,6,5,6,5,4,4,4,4,3,3,3
1,0,6,8,9,7,6,8,5,4,3,4,3,3,4,4,6,6,5,2,4
2,1,7,7,6,6,6,7,3,3,5,5,6,3,7,6,6,6,6,7,7
3,3,5,4,8,5,6,4,7,7,7,5,7,7,7,7,7,7,7,7,7
4,4,3,7,4,4,4,5,4,5,6,5,7,7,6,7,7,7,7,7,7
5,3,2,3,3,4,4,5,7,7,7,7,7,7,7,7,7,7,7,7,7
6,4,2,3,5,5,5,7,7,7,7,7,7,7,7,7,7,7,7,7,7
7,4,3,3,3,4,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
8,4,4,5,4,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
9,4,5,4,5,5,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7


# talking about the bridge

In [12]:
def bridge_stats(dir):
    """
    counting the number of steered sentences that talk about the golden gate bridge 
    """
    print("showing result for directory", dir)
    grid_bridge = pd.DataFrame(0, index=range(32),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(32),columns=range(20))
    grid_fl = pd.DataFrame(index=range(32),columns=range(20))
    for file_name in os.listdir(f"{result_path}{dir}/"):
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["bridge"].value_counts():
                grid_bridge.loc[layer, coeff-1] = list_dict["bridge"].value_counts()[1]
            else:
                grid_bridge.loc[layer, coeff-1] = 0
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object
    
    print("count of sentences talking about the bridge ↑")
    display(grid_bridge.style.background_gradient(cmap='Blues'))
    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds'))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )

In [13]:
for dir in dirs_bridge:
    bridge_stats(dir)

showing result for directory gemini_bridge_llama_bridge+_fl_hpt
count of sentences talking about the bridge ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,10,11,9,10,10,8,6,5,2,2,0,0,0,0,0,0,0,0,1
1,0,13,14,12,15,15,15,11,5,5,4,6,9,10,10,13,14,11,12,11
2,0,12,11,12,12,14,8,7,6,6,8,7,9,15,17,15,16,17,19,20
3,0,3,5,7,13,11,14,14,17,17,18,18,16,15,11,14,12,12,16,13
4,0,5,6,2,0,0,5,7,14,18,17,19,15,15,14,13,16,16,18,17
5,0,2,3,3,4,4,10,14,18,16,18,16,17,19,18,18,17,17,18,19
6,0,4,4,1,2,4,10,11,7,11,15,14,16,19,19,19,18,17,18,18
7,0,3,1,0,0,2,9,10,13,15,15,14,14,16,18,17,18,20,20,19
8,0,4,1,0,1,7,8,9,10,10,11,8,10,9,11,10,9,11,12,12
9,0,2,0,0,1,1,8,9,10,9,11,10,15,14,14,15,14,14,16,16


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,11,14,18,18,19,20,20,20,20,20,19,20,20,20,20,20,20,20,20
1,7,4,14,16,19,19,20,19,20,18,20,20,20,20,20,20,20,20,20,20
2,8,6,8,13,12,11,10,11,10,11,11,14,12,18,19,20,20,20,20,20
3,10,7,10,8,11,8,6,5,6,10,3,7,10,16,20,19,20,20,20,19
4,7,7,10,10,10,13,8,12,18,20,19,20,19,19,20,20,20,20,20,20
5,8,10,9,7,15,13,13,15,19,20,20,20,19,19,19,19,20,20,20,20
6,5,11,8,8,4,9,14,14,16,17,16,17,19,19,18,18,18,19,19,19
7,9,10,10,9,13,14,16,16,14,17,18,19,19,18,19,20,20,20,20,19
8,5,7,10,4,7,11,12,15,18,17,16,18,18,19,20,20,20,19,19,19
9,10,6,8,11,8,9,9,11,14,17,18,18,18,18,19,19,19,18,17,18


average perplexity of continuations ↓


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,10,12,15,13,9,6,2,0,0,0,0,0,0,0,0,0,0,0,0
1,2,9,8,5,1,0,0,0,0,1,0,0,1,0,0,0,0,0,2,0
2,3,6,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,3,1,0,0,0,0,1,1,1,2,1,1,2,0,0,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
5,0,1,0,0,1,0,0,3,1,0,0,0,0,0,0,0,0,0,0,0
6,4,0,1,0,0,0,0,1,2,1,1,0,1,3,2,2,1,1,1,1
7,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
8,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
9,2,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,16,8,7,8,9,8,11,13,14,13,12,14,16,16,17,18,18,17,18,16
1,15,15,18,18,19,16,17,17,12,16,12,15,13,13,11,15,11,7,9,16
2,14,20,19,19,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
3,19,19,18,18,18,17,18,16,13,12,15,17,18,11,15,14,12,10,18,12
4,17,16,17,19,18,18,17,18,20,19,19,19,18,19,20,18,18,20,17,18
5,12,13,10,9,11,9,16,17,16,17,18,20,17,19,20,19,19,20,19,18
6,12,18,19,17,17,12,13,15,14,12,12,10,11,13,11,15,13,10,9,10
7,19,20,19,18,18,20,20,20,20,19,20,20,19,20,20,19,19,17,17,17
8,16,19,20,20,20,20,20,20,20,20,20,20,20,20,20,18,20,20,19,20
9,14,20,20,20,20,20,20,20,20,20,20,20,19,18,19,20,20,20,20,20


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.97,494.51,296.17,225.47,210.17,47.56,28.31,27.89,3.77,3.64,3.78,3.62,3.77,46.48,46.12,126.80,3.33,127.15,3.36,3.44
1,3.33,33.35,5.61,175.74,262.68,149.32,"1,452.76",17.85,16.63,"4,660.47","4,770.87","5,926.78","15,392.95","987,048.46","986,853.39",573.25,234.17,"2,982.71",51.55,"52,251.56"
2,3.33,5.00,"33,590,060.79",315.85,14.73,5.72,4.83,4.97,5.02,5.04,4.70,4.13,4.36,3.99,4.41,4.42,4.31,4.24,4.65,4.88
3,3.79,"6,591.23","93,692.07","409,339.55","649,212.78","326,473.98","230,027.11","813,321.22","3,708.39",227.78,"148,554.29",43.25,"1,039.48",121.35,40.12,46.92,39.62,31.11,77.50,42.85
4,62.04,"387,462.33","33,894,251.21","8,679.78","3,397.20","79,052,210.68","79,112,144.21",346.05,5.50,13.22,11.78,22.95,"2,865.32",11.89,20.63,13.97,24.78,8.58,9.35,9.31
5,393.78,"49,822,053.89","831,550,021.92","234,392,746.12","165,622.71","153,957.82","94,350.79","716,918,407.26","245,241.37","258,913.92","379,087.10","161,820.69","85,223.57","90,785.95","391,649.54","340,159.04","340,621.24","338,409.05","347,785.81","414,223.41"
6,3.24,"30,018,741.02","71,813,186.75","679,912.73",598.20,"5,221,587.39","3,729.09","12,022.01","15,882.39","103,522.08","385,965.02","1,227,976.14","555,477.67","841,378,270.78","7,898,921,143.35","884,507.23","44,847,064,121.40","63,644.01","7,422,031.73","1,333,124.63"
7,"500,486,333.09","6,942,131,683.03","33,151,323,098.99","108,404,197,491.83","14,321,419,907.15","36,523,756,415.02","8,600,194,180.03","1,468,489,784.26","1,032,113,335.12","3,221.86",245.50,117.42,244.41,29.35,159.29,"1,851.97","3,306.72","2,205,931.81","1,389,743.43","4,580.83"
8,61.19,"10,637,479,908.06","31,519,641,978.56","28,352,481,103.59","29,218,225,779.78","28,827,661,581.18","960,303,093.92","960,299,456.02","1,953.57",18.75,28.00,203.35,58.30,45.86,62.86,"4,501,862.95",162.21,270.36,210.63,"2,951.23"
9,2.88,27.64,515.59,633.18,825.53,"7,640,151,337.29",737.96,"716,918,036.24",280.23,"161,070,746.31","6,334,572,532.98","8,911,927,935.49","7,895,775,744.11","7,895,776,546.85","8,612,694,479.68","8,612,694,542.01","8,612,694,230.17","7,895,778,440.36","7,640,155,431.84","2,391,697,200.70"


# temperature=1
all the results above are for when temperature = 0
below are additional analysis for when temperature = 1, which includes
- steering with a pair of phrases with one white space prepended to each phrase
- steering with a pair of sentences

In [ ]:
baseline_llama_temp1 = "gemini_base_llama_fl_senti.json"
baseline_opt_temp1 = "gemini_base_opt_fl_senti.json"

result_path = "/scratch/fmeng/ActAdd/results/"
dirs_2pos_temp1 = [
    "gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_llama_senti+_fl_temp_0_hpt", 
    "gemini_sent_2pos_opt_senti+_fl_temp_0_hpt"]

dirs_2neg_temp1 = [
    "gemini_2neg_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2neg_llama_senti+_fl_temp_0_hpt",
    "gemini_sent_2neg_opt_senti+_fl_temp_0_hpt"]

